In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path
from dask import delayed, compute
import multiprocessing

# Use this notebook to create Zarr stores for each of the datasets
Should you wish not to use the default locations set by the tutorial notebook, you will have to provide the path to the netcdf files and the path to where you want the zarr store to exist.<br>

It is worth noting that each netcdf file contains all data for a single station, and there are two functions defined below that are necessary for converting our station data to zarr:
- `preprocess_station`
- `process_all_to_zarr`



There is a final function `load_combined_dataset` that is used to verify the conversion succeeded and that all station data can be loaded together.


# Preprocess NetCDF Files
Purpose:
- Opens a single NetCDF file, standardizes it, and prepares it for analysis.

Key steps:
- Loads the NetCDF file as an xarray Dataset.
- Drops the input_station_id variable if present (to avoid object dtype issues).
- Assigns a station_id coordinate from the file’s attributes or filename.
- Reindexes the time dimension to a common hourly range (ensuring all stations align in time).
- Returns the cleaned dataset.


In [2]:
def preprocess_station(file_path, date_range):
    """Open and preprocess a single NetCDF file."""
    ds = xr.open_dataset(file_path)

    # Drop redundant or uninformative station ID variables if present
    for var in ['input_station_id', 'station_id']:
        if var in ds:
            ds = ds.drop_vars(var)

    # Assign station ID from attributes or filename
    station_id = ds.attrs.get("station_id", file_path.stem)
    ds = ds.expand_dims({"station": [station_id]})

    # Promote lat/lon/elevation to coordinates (if not already)
    for coord in ["latitude", "longitude", "elevation"]:
        if coord in ds and coord not in ds.coords:
            ds = ds.set_coords(coord)

    for coord in ["latitude", "longitude", "elevation"]:
        if coord in ds and "coordinate_length" in ds[coord].dims:
            ds[coord] = ds[coord].squeeze("coordinate_length")

    # Reindex time to common range
    target_time = pd.date_range(date_range[0], date_range[1], freq='h')
    ds = ds.reindex(time=target_time)

    return ds

    #Explain reason for reindexing and cutting out pre 1970 poor data quality
    

The HadISD NetCDF files store latitude, longitude, and elevation as coordinates with a singleton coordinate_length dimension. When merging multiple stations, these become coordinates of shape (station, coordinate_length). To ensure they are always available as coordinates (and not lost when selecting variables), we explicitly promote them with set_coords. After merging, we remove the unnecessary coordinate_length dimension, resulting in 1D auxiliary coordinates of shape (station,) for each station.

# Convert NetCDF to Zarr

Purpose:
- Batch-processes all NetCDF files in a directory, converting each to a Zarr store.

Key steps:
- Iterates through all .nc files in the input directory.
- Applies the preprocess_station function to each file.
- Saves each processed dataset as an individual Zarr store in the output directory.


In [3]:
def process_all_to_zarr(netcdf_dir, zarr_output_dir, date_range, scheduler='threads', num_workers=None):
    """Parallel NetCDF to Zarr conversion using dask.delayed. Returns both success boolean and status message."""
    netcdf_dir = Path(netcdf_dir)
    zarr_output_dir = Path(zarr_output_dir)
    zarr_output_dir.mkdir(parents=True, exist_ok=True)

    netcdf_files = list(netcdf_dir.glob("*.nc"))
    zarr_files = set(f.stem for f in zarr_output_dir.glob("*.zarr"))

    tasks = []
    skipped = 0

    def convert_netcdf_to_zarr(nc_file, out_path, date_range):
        try:
            ds = preprocess_station(nc_file, date_range)
            ds.to_zarr(str(out_path), mode='w')
            return (True, f"Converted: {nc_file.name} → {out_path.name}")
        except Exception as e:
            return (False, f"Failed on {nc_file.name}: {e}")

    for nc_file in netcdf_files:
        zarr_name = nc_file.stem
        out_path = zarr_output_dir / f"{zarr_name}.zarr"
        if zarr_name in zarr_files:
            # print(f"Zarr file already exists for {nc_file.name}: {out_path.name}. Skipping.")
            skipped += 1
            continue
        tasks.append(delayed(convert_netcdf_to_zarr)(nc_file, out_path, date_range))

    if not tasks:
        print(f"No new NetCDF files to convert. {skipped} already present.")
        return

    if num_workers is None:
        num_workers = multiprocessing.cpu_count() // 2

    print(f"Starting Dask parallel conversion with {num_workers} workers...")
    results = compute(*tasks, scheduler=scheduler, num_workers=num_workers)
    for success, msg in results:
        print(msg)
    converted = sum(success for success, _ in results)
    print(f"Conversion complete. {converted} new stations converted, {skipped} already present.")

# Load all individual Zarr stores into a single xarray Dataset
Purpose:
- Loads all individual Zarr stores and combines them into a single xarray Dataset for analysis.

Key steps:
- Finds all .zarr stores in the specified directory.
- Uses xr.open_mfdataset to open and concatenate them along the station dimension.
- Returns the combined dataset, ready for further analysis.

In [4]:
def load_combined_dataset(zarr_dir):
    # Open all Zarr stores together
    zarr_paths = list(Path(zarr_dir).glob("*.zarr"))

    # Combine along station dimension
    ds = xr.open_mfdataset(
        zarr_paths,
        combine="nested",
        concat_dim="station",
        parallel=True,
        engine="zarr"
    )
    return ds

Combining data like this is far quicker and more resource efficient than using NetCDF files directly.

# Execute the Pre-processing and Conversion to Zarr 
We can run the `Data_Config.ipynb` to set the following:
- Date range to reindex the data
- Path to NetCDFs that need converting to Zarr
- Output location of the Zarr store

In [5]:
# ruff: noqa: F821
%run Data_Config.ipynb

In [6]:
# Run parallel NetCDF-to-Zarr
process_all_to_zarr(str(input_dir), str(zarr_output_dir), DATE_RANGE, scheduler='processes')

Starting Dask parallel conversion with 4 workers...
Converted: hadisd.3.4.0.2023f_19310101-20240101_722866-23122.nc → hadisd.3.4.0.2023f_19310101-20240101_722866-23122.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_722780-23183.nc → hadisd.3.4.0.2023f_19310101-20240101_722780-23183.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_722122-03050.nc → hadisd.3.4.0.2023f_19310101-20240101_722122-03050.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_529570-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_529570-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_722031-63839.nc → hadisd.3.4.0.2023f_19310101-20240101_722031-63839.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_722334-03976.nc → hadisd.3.4.0.2023f_19310101-20240101_722334-03976.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_515730-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_515730-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_544830-99999.nc → hadisd.3.4.0.2023f_19310101-2024010

Show the combined dataset from the Zarr stores

In [7]:
ds_combined = load_combined_dataset(zarr_output_dir)
ds_combined

<xarray.Dataset> Size: 327GB
Dimensions:                (station: 770, time: 473352, flagged: 19, test: 71,
                            reporting_v: 19, reporting_t: 1116, reporting_2: 2)
Coordinates:
    elevation              (station) float64 6kB 1.676e+03 401.0 ... 117.7 36.0
    latitude               (station) float64 6kB 38.47 42.33 ... 45.93 30.48
    longitude              (station) float64 6kB 102.2 120.7 ... 126.6 -87.19
  * station                (station) object 6kB '526740-99999' ... '722223-13...
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-3...
Dimensions without coordinates: flagged, test, reporting_v, reporting_t,
                                reporting_2
Data variables: (12/25)
    cloud_base             (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    dewpoints              (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 55GB dask.array<chunksize=(1, 29585, 3), meta=np.ndarray>
    high_cloud_cover       (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    low_cloud_cover        (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    mid_cloud_cover        (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    ...                     ...
    stnlp                  (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    temperatures           (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    total_cloud_cover      (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    wind_gust              (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    winddirs               (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    windspeeds             (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
Attributes: (12/39)
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    acknowledgement:             RJHD was supported by the Joint BEIS/Defra M...
    cdm_data_type:               station
    creator_email:               robert.dunn@metoffice.gov.uk
    creator_name:                Robert Dunn
    ...                          ...
    station_id:                  526740-99999
    station_information:         Where station is a composite the station id ...
    summary:                     Quality-controlled, sub-daily, station datas...
    time_coverage_end:           2002-05-13T09:00Z
    time_coverage_start:         1964-01-01T00:00Z
    title:                       HadISD